# TP : SVM — Corrigé
Ce notebook contient la solution complète de chaque cellule `TODO`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from matplotlib.colors import ListedColormap


In [ ]:
def plot_decision_boundary_with_margins(clf, X, y, h=0.02, title="Decision Boundary"):
    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    cmap_bold  = ListedColormap(['#FF0000', '#0000FF'])

    x_min, x_max = X[:,0].min() - 1, X[:,0].max() + 1
    y_min, y_max = X[:,1].min() - 1, X[:,1].max() + 1

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    Z = clf.predict(grid).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)

    if hasattr(clf, "decision_function"):
        df = clf.decision_function(grid).reshape(xx.shape)
        plt.contour(xx, yy, df, levels=[0], colors='k', linewidths=2)
        plt.contour(xx, yy, df, levels=[-1, 1],
                    colors='k', linestyles='--', linewidths=1.4)

    plt.scatter(X[:,0], X[:,1], c=y, cmap=cmap_bold, s=30, edgecolors='k')

    if hasattr(clf, "support_vectors_"):
        plt.scatter(
            clf.support_vectors_[:,0],
            clf.support_vectors_[:,1],
            s=120, linewidth=1,
            facecolors='none', edgecolors='yellow',
            label='Support Vectors'
        )

    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.show()


## PARTIE 1 — SVM linéaire sur Iris

In [ ]:
iris = datasets.load_iris()

# On garde uniquement les deux dernières caractéristiques (longueur et largeur des pétales)
X = iris.data[:, 2:4]
y = iris.target

# Classification binaire : Versicolor (1) vs Virginica (2)
mask = y != 0
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)


In [ ]:
for C in [1, 100]:
    print("----- Test avec C =", C, "-----")
    model = SVC(kernel='linear', C=C)
    model.fit(X_train, y_train)
    print("Accuracy (test) :", model.score(X_test, y_test))
    plot_decision_boundary_with_margins(model, X_train, y_train,
                                        title=f"SVM linéaire, C = {C}")


## PARTIE 2 — SVM non linéaire sur Moons

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=0)

plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons)
plt.title("Moons Dataset")
plt.show()

# Découpage train/test pour évaluer le modèle sur des données non vues
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=0
)


In [ ]:
model_rbf = SVC(kernel='rbf', gamma='scale')
model_rbf.fit(Xm_train, ym_train)
print("Accuracy RBF (test) :", model_rbf.score(Xm_test, ym_test))
plot_decision_boundary_with_margins(model_rbf, X_moons, y_moons,
                                    title="SVM RBF sur Moons")


In [ ]:
model_lin = SVC(kernel='linear', C=1)
model_lin.fit(Xm_train, ym_train)
print("Accuracy linéaire (test) :", model_lin.score(Xm_test, ym_test))
plot_decision_boundary_with_margins(model_lin, X_moons, y_moons,
                                    title="SVM linéaire sur Moons")


## PARTIE 3 — Astuce du noyau

In [ ]:
kernels = ['poly', 'rbf', 'sigmoid']

for k in kernels:
    print("----- Kernel =", k, "-----")
    model = SVC(kernel=k, gamma='scale')
    model.fit(Xm_train, ym_train)
    print("Accuracy (test) :", model.score(Xm_test, ym_test))
    plot_decision_boundary_with_margins(model, X_moons, y_moons,
                                        title=f"Noyau {k}")


## PARTIE 4 — Corrigé des questions théoriques

1. **Échec du SVM linéaire** : il trace un hyperplan, donc une frontière droite. Si les classes ne sont pas séparables par une droite (comme les deux lunes), il ne peut pas bien les séparer.
2. **Rôle de C** : il contrôle le compromis entre marge large et erreurs de classification. Un C grand pénalise fortement les erreurs (risque de surapprentissage) ; un C petit accepte davantage d'erreurs pour une marge plus large.
3. **Rôle de gamma (RBF)** : il règle la portée de l'influence d'un point d'entraînement. Un gamma grand rend la frontière très locale et complexe (surapprentissage) ; un gamma petit la rend plus lisse.
4. **Astuce du noyau** : elle permet de calculer un produit scalaire dans un espace de grande dimension (voire de dimension infinie, pour le RBF) sans jamais calculer explicitement la transformation. On obtient ainsi une frontière non linéaire dans l'espace d'origine, à faible coût de calcul.
5. **Sensibilité à l'échelle** : la marge et les distances entre points dépendent des unités. Une variable avec de grandes valeurs domine le calcul. Il faut donc standardiser les données (par exemple avec `StandardScaler`).
6. **SVM linéaire ou à noyau** : on choisit le linéaire quand les données sont déjà presque linéairement séparables, quand le nombre de variables est grand (texte, par exemple), ou quand on veut un modèle rapide et interprétable. Le noyau est préférable quand la frontière est clairement non linéaire.
